# Cube Nano - SegFormer-B0 training on 95-Cloud

This Colab notebook trains the RGB, two-class SegFormer-B0 cloud segmenter using the repository pipeline rather than a duplicate notebook-only training loop. It preserves native scene dimensions during training and validation with batch size 1, then exports the fixed runtime graph with input `[1, 3, 256, 256]`.

Before running, select a GPU runtime and add the `KAGGLE_API_TOKEN` secret in Colab. Set `repo_ref` to an immutable commit that contains the SegFormer files before a reproducible run.

> The default mode is `research_baseline`. It creates a checkpoint and evidence bundle, but does not mark a deployment release as valid. The local SegFormer implementation currently has no pinned pretrained-weight loader, and the radiometric/target gates in the integration plan remain separate release requirements.


## 1. Install the Colab dependencies

Colab supplies the CUDA-enabled PyTorch build. This cell deliberately does not install the CPU lockfile, which would replace that build.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

required_packages = {
    'tifffile': 'tifffile',
    'tqdm': 'tqdm',
    'yaml': 'PyYAML',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'kaggle': 'kaggle',
    'pytest': 'pytest',
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Training will stop later unless a GPU runtime is selected.')


## 2. Configure the run and source revision

Use a commit SHA for `repo_ref` when reproducing a candidate. `raw_audit` is intentionally explicit: a release-candidate run stops when the required radiometric fields are still marked `UNVERIFIED`.


In [ ]:
import datetime as dt
import hashlib
import json
import os
import platform
import shlex
import shutil
from pathlib import Path

CFG = {
    'repo_url': 'https://github.com/hoxuanphu/cube_nano.git',
    'repo_ref': 'feature/segformer-b0-integration',
    'kaggle_slug': 'sorour/95cloud-cloud-segmentation-on-satellite-images',
    'data_root_override': None,
    'raw_on_drive': True,
    'remove_kaggle_archive': True,
    'cleanup_stale_local_workspace': False,
    'cleanup_local_after_bundle': True,
    'run_mode': 'research_baseline',  # research_baseline or release_candidate
    'rebuild_processed_data': False,
    'move_split': True,  # Avoid a second processed copy; split_dataset hashes before moving.
    'run_regression_tests': True,
    'seed': 42,
    'cloud_ratio_threshold': 0.10,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    'epochs': 50,
    'learning_rate': 6e-5,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,
    'early_stopping_patience': 12,
    'use_amp': True,
    'max_false_clear_rate': 0.05,
    'threshold_start_bp': 1000,
    'threshold_stop_bp': 10000,
    'threshold_step_bp': 100,
    'bootstrap_samples': 1000,
}

raw_audit = {
    'sensor_id': 'UNVERIFIED',
    'platform_id': 'UNVERIFIED',
    'product_type': 'UNVERIFIED',
    'processing_level': 'UNVERIFIED',
    'units': 'UNVERIFIED',
    'scale_offset': 'UNVERIFIED',
    'nodata': 'UNVERIFIED',
    'saturation': 'UNVERIFIED',
    'gsd': 'UNVERIFIED',
    'band_order': ['red', 'green', 'blue'],
    'ground_truth_encoding_confirmed': False,
    'ground_truth_clear_values': [0],
    'ground_truth_cloud_values': [1, 255],
    'invalid_ground_truth_values': [],
}

if CFG['run_mode'] not in {'research_baseline', 'release_candidate'}:
    raise ValueError('run_mode must be research_baseline or release_candidate')
if not 0 <= CFG['max_false_clear_rate'] <= 1:
    raise ValueError('max_false_clear_rate must be in [0, 1]')
if not 0 <= CFG['val_ratio'] < 1 or not 0 <= CFG['test_ratio'] < 1:
    raise ValueError('split ratios must be in [0, 1)')
if CFG['val_ratio'] + CFG['test_ratio'] >= 1:
    raise ValueError('validation and test ratios must leave a train split')

CONTENT = Path('/content')
PROJECT = CONTENT / 'cube_nano'
RAW = CONTENT / '95cloud_kaggle'
RUN = CONTENT / 'segformer_95cloud_run'
PROCESSED = RUN / 'data' / 'processed'
PROCESSED_ALL = PROCESSED / 'all'
CHECKPOINTS = RUN / 'checkpoints'
RESULTS = RUN / 'results'
CONTRACTS = RUN / 'contracts'
ARTIFACTS = RUN / 'artifacts'
DELIVERABLES = RUN / 'deliverables'
for directory in (RUN, CHECKPOINTS, RESULTS, CONTRACTS, ARTIFACTS, DELIVERABLES):
    directory.mkdir(parents=True, exist_ok=True)

print(json.dumps(CFG, indent=2, sort_keys=True))
print('Run directory:', RUN)


## 3. Mount Google Drive for persistent checkpoints

The training workspace remains on `/content`; this cell mounts Drive and creates a persistent run directory. With `raw_on_drive=True`, raw TIFFs are downloaded and extracted under Drive, the checkpoint is copied there immediately after training, and the final evidence bundle is synced there as well.


In [ ]:
from google.colab import drive

DRIVE_MOUNT = '/content/drive'
drive.mount(DRIVE_MOUNT, force_remount=False)
DRIVE_ROOT = Path(DRIVE_MOUNT) / 'MyDrive' / 'cube_nano' / 'segformer_95cloud'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_DELIVERABLES = DRIVE_ROOT / 'deliverables'
def ensure_drive_directory(directory):
    directory = Path(directory)
    if directory.exists():
        if not directory.is_dir():
            raise NotADirectoryError(f'Drive path exists but is not a directory: {directory}')
        return directory
    directory.mkdir(parents=True)
    return directory

for directory in (DRIVE_CHECKPOINTS, DRIVE_RESULTS, DRIVE_DELIVERABLES):
    ensure_drive_directory(directory)

def save_to_drive(source, destination):
    source = Path(source)
    destination = Path(destination)
    if not source.is_file():
        raise FileNotFoundError(f'Cannot sync missing file to Drive: {source}')
    ensure_drive_directory(destination.parent)
    shutil.copy2(source, destination)
    return destination

LOCAL_RAW = RAW
if CFG['raw_on_drive'] and not CFG['data_root_override']:
    RAW = DRIVE_ROOT / 'raw_95cloud'
    ensure_drive_directory(RAW)
    print('Raw dataset will be stored on Drive:', RAW)
else:
    print('Raw dataset location:', RAW)

def disk_report(label):
    usage = shutil.disk_usage(CONTENT)
    gib = 1024 ** 3
    print(f'{label}: {usage.free / gib:.1f} GiB free / {usage.total / gib:.1f} GiB total')

def remove_content_path(path):
    path = Path(path).resolve()
    content_root = CONTENT.resolve()
    if path == content_root or content_root not in path.parents:
        raise ValueError(f'Refusing to remove a path outside the Colab content directory: {path}')
    if path.exists():
        shutil.rmtree(path)
        print('Removed stale local path:', path)

if CFG['cleanup_stale_local_workspace']:
    remove_content_path(LOCAL_RAW)
    remove_content_path(PROCESSED)
else:
    print('Set cleanup_stale_local_workspace=True and rerun this cell to remove a previous failed local run.')
for directory in (RUN, CHECKPOINTS, RESULTS, CONTRACTS, ARTIFACTS, DELIVERABLES):
    directory.mkdir(parents=True, exist_ok=True)
disk_report('After Drive setup')

print('Drive persistence root:', DRIVE_ROOT)


## 4. Clone the exact repository revision

The notebook checks out the requested ref in detached mode and writes the resolved commit to the run provenance. It refuses a non-Git directory at the clone path instead of deleting it.


In [ ]:
def run_command(label, command, *, cwd=None):
    command = [str(item) for item in command]
    print(f'[{label}]')
    print('$', shlex.join(command))
    completed = subprocess.run(command, cwd=str(cwd) if cwd else None, check=False)
    if completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, command)

if PROJECT.exists() and not (PROJECT / '.git').is_dir():
    raise FileExistsError(f'Clone target is not a Git repository: {PROJECT}')
if not PROJECT.exists():
    run_command('Clone repository', ['git', 'clone', '--no-checkout', CFG['repo_url'], PROJECT])
run_command('Fetch requested ref', ['git', '-C', PROJECT, 'fetch', '--depth', '1', 'origin', CFG['repo_ref']])
run_command('Checkout requested ref', ['git', '-C', PROJECT, 'checkout', '--detach', 'FETCH_HEAD'])

required_entrypoints = (
    'src/data/preprocess_95cloud.py',
    'src/data/split_dataset.py',
    'src/data/segmentation_dataset.py',
    'src/models/segformer_b0.py',
    'src/train_segmentation.py',
    'src/eval_segmentation.py',
    'src/export_segformer_onnx.py',
    'sat_ai/segformer_model_manifest.yaml',
)
missing_entrypoints = [relative for relative in required_entrypoints if not (PROJECT / relative).is_file()]
if missing_entrypoints:
    raise FileNotFoundError(
        'The selected repo_ref does not contain the SegFormer pipeline: ' + ', '.join(missing_entrypoints)
    )

source_revision = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True
).strip()
source_provenance = {
    'repository_url': CFG['repo_url'],
    'requested_ref': CFG['repo_ref'],
    'resolved_commit': source_revision,
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
}
(RESULTS / 'source_provenance.json').write_text(
    json.dumps(source_provenance, indent=2, sort_keys=True), encoding='utf-8'
)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Resolved commit:', source_revision)


## 5. Download and locate 95-Cloud

The Kaggle token is read from Colab Secrets and stays in the process environment only. Set `data_root_override` when the raw TIFF files are already mounted from Drive.


In [ ]:
from google.colab import userdata
from src.data.preprocess_95cloud import discover_scene_files

def has_tiff_files(directory):
    directory = Path(directory)
    return directory.is_dir() and any(
        path.is_file() and path.suffix.lower() in {'.tif', '.tiff'}
        for path in directory.rglob('*')
    )

if CFG['data_root_override']:
    raw_root = Path(CFG['data_root_override']).expanduser().resolve()
    if not raw_root.is_dir():
        raise FileNotFoundError(f'data_root_override does not exist: {raw_root}')
else:
    if not RAW.exists():
        RAW.mkdir(parents=True)
    elif not RAW.is_dir():
        raise NotADirectoryError(f'Raw Drive path exists but is not a directory: {RAW}')
    if not has_tiff_files(RAW):
        kaggle_token = userdata.get('KAGGLE_API_TOKEN')
        if not kaggle_token:
            raise RuntimeError('Colab Secret KAGGLE_API_TOKEN is missing or empty')
        os.environ['KAGGLE_API_TOKEN'] = kaggle_token
        run_command(
            'Download 95-Cloud from Kaggle',
            ['kaggle', 'datasets', 'download', '-d', CFG['kaggle_slug'], '-p', RAW, '--unzip'],
        )
    if CFG['remove_kaggle_archive']:
        for archive in RAW.glob('*.zip'):
            archive.unlink()
            print('Removed extracted Kaggle archive:', archive)
    raw_root = RAW

def locate_dataset_root(root):
    root = Path(root).resolve()
    tiff_parents = {path.parent for path in root.rglob('*') if path.suffix.lower() in {'.tif', '.tiff'}}
    candidates = [root]
    for parent in sorted(tiff_parents, key=lambda value: (len(value.parts), str(value))):
        candidates.extend([parent, *parent.parents[:5]])
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in seen or not candidate.is_dir():
            continue
        seen.add(candidate)
        try:
            scenes = discover_scene_files(candidate, channels=3)
        except (FileNotFoundError, ValueError):
            continue
        if scenes:
            return candidate, scenes
    raise FileNotFoundError(f'Could not locate complete RGB plus GT scenes under {root}')

DATA_ROOT, SCENE_FILES = locate_dataset_root(raw_root)
print('Dataset root:', DATA_ROOT)
print('Complete RGB plus GT scenes:', len(SCENE_FILES))
disk_report('After raw dataset resolution')


## 6. Freeze the current contracts and audit the raw pairs

This creates a raw-file manifest, validates RGB/GT shape and dtype consistency, records observed GT values, and writes the declared validity policy. It does not invent sensor or radiometric metadata: fill `raw_audit` before switching to `release_candidate`.


In [ ]:
import numpy as np
import tifffile
import yaml

segformer_manifest = yaml.safe_load((PROJECT / 'sat_ai/segformer_model_manifest.yaml').read_text(encoding='utf-8'))
acceptance_profile = yaml.safe_load((PROJECT / 'sat_ai/acceptance_profile.yaml').read_text(encoding='utf-8'))
if segformer_manifest.get('model_task') != 'semantic_cloud_segmentation':
    raise ValueError('SegFormer manifest does not declare semantic_cloud_segmentation')
if segformer_manifest['input_spec']['band_order'] != ['red', 'green', 'blue']:
    raise ValueError('The notebook only supports canonical RGB band order')
if segformer_manifest['input_spec']['input_shape'] != [None, 3, 256, 256]:
    raise ValueError('Runtime input contract must remain [null, 3, 256, 256]')
if segformer_manifest['output']['model_output']['shape'] != [1, 2, 64, 64]:
    raise ValueError('Unexpected SegFormer logits contract')
if acceptance_profile['quality']['max_false_clear_rate'] != CFG['max_false_clear_rate']:
    raise ValueError('CFG max_false_clear_rate must match the acceptance profile')

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

declared_clear = {int(value) for value in raw_audit['ground_truth_clear_values']}
declared_cloud = {int(value) for value in raw_audit['ground_truth_cloud_values']}
declared_invalid = {int(value) for value in raw_audit['invalid_ground_truth_values']}
if declared_clear & declared_cloud or declared_clear & declared_invalid or declared_cloud & declared_invalid:
    raise ValueError('Ground-truth clear, cloud, and invalid value sets must be disjoint')
decoder_clear = {0} - declared_invalid
decoder_cloud = {1, 255} - declared_invalid
if declared_clear != decoder_clear or declared_cloud != decoder_cloud:
    raise ValueError(
        'raw_audit encoding must match preprocess_95cloud.decode_ground_truth for this run'
    )
allowed_gt_values = {0, 1, 255, *declared_invalid}
raw_scenes = []
observed_gt_values = set()
for scene_id, files in SCENE_FILES.items():
    bands = []
    expected_shape = None
    for band in ('red', 'green', 'blue'):
        path = Path(files[band])
        array = np.asarray(tifffile.imread(path))
        if array.ndim != 2 or array.dtype != np.uint16:
            raise ValueError(f'{scene_id}/{band} must be a uint16 2D TIFF, got {array.dtype} {array.shape}')
        expected_shape = expected_shape or array.shape
        if array.shape != expected_shape:
            raise ValueError(f'RGB shape mismatch in scene {scene_id}')
        bands.append({
            'band': band,
            'path': str(path.relative_to(DATA_ROOT)),
            'sha256': sha256_file(path),
            'dtype': str(array.dtype),
            'shape': [int(value) for value in array.shape],
            'min': int(array.min()),
            'max': int(array.max()),
        })
    gt_path = Path(files['gt'])
    ground_truth = np.asarray(tifffile.imread(gt_path))
    if ground_truth.ndim != 2 or ground_truth.shape != expected_shape:
        raise ValueError(f'Ground truth shape mismatch in scene {scene_id}')
    values = sorted(int(value) for value in np.unique(ground_truth))
    unexpected = sorted(set(values) - allowed_gt_values)
    if unexpected:
        raise ValueError(
            f'Unaudited GT values in {scene_id}: {unexpected}. Update the audit before preprocessing.'
        )
    observed_gt_values.update(values)
    raw_scenes.append({
        'scene_id': scene_id,
        'bands': bands,
        'ground_truth': {
            'path': str(gt_path.relative_to(DATA_ROOT)),
            'sha256': sha256_file(gt_path),
            'dtype': str(ground_truth.dtype),
            'shape': [int(value) for value in ground_truth.shape],
            'values': values,
        },
    })

raw_manifest = {
    'schema_version': 1,
    'dataset_root': str(DATA_ROOT),
    'scene_count': len(raw_scenes),
    'observed_ground_truth_values': sorted(observed_gt_values),
    'declared_audit': raw_audit,
    'scenes': raw_scenes,
}
raw_manifest['raw_manifest_id'] = canonical_hash(raw_manifest)
raw_manifest_path = RESULTS / 'raw_dataset_audit.json'
raw_manifest_path.write_text(json.dumps(raw_manifest, indent=2, sort_keys=True), encoding='utf-8')

unverified = [
    field for field in ('sensor_id', 'platform_id', 'product_type', 'processing_level', 'units',
                  'scale_offset', 'nodata', 'saturation', 'gsd')
    if raw_audit[field] == 'UNVERIFIED'
]
if CFG['run_mode'] == 'release_candidate' and (unverified or not raw_audit['ground_truth_encoding_confirmed']):
    raise RuntimeError(
        'Release-candidate mode requires a completed raw_audit. Unverified fields: ' + ', '.join(unverified)
    )

print('Raw manifest ID:', raw_manifest['raw_manifest_id'])
print('Observed GT values:', raw_manifest['observed_ground_truth_values'])
if CFG['run_mode'] == 'research_baseline':
    print('Research baseline mode: radiometry and pretrained-artifact release gates remain open.')


## 7. Run the SegFormer regression tests

The targeted suite covers mask/validity semantics, native-size loss, threshold selection, postprocess parity, products, and the fixed graph contract.


In [ ]:
if CFG['run_regression_tests']:
    run_command(
        'SegFormer regression tests',
        [sys.executable, '-m', 'pytest', 'tests/test_segformer_integration.py', '-q'],
        cwd=PROJECT,
    )
else:
    print('Regression tests skipped by configuration.')


## 8. Preprocess native scenes and split at scene level

Preprocessing stores an RGB image, cloud target, validity mask, and raw GT with the same filename. The default move-based split avoids retaining a second `all/` copy; `split_dataset.py` computes the lineage hash before moving files.


In [ ]:
def split_ready():
    manifest = PROCESSED / 'scene_split_manifest.json'
    return manifest.is_file() and all(
        (PROCESSED / split / 'masks').is_dir()
        and any((PROCESSED / split / 'masks').glob('*.npy'))
        for split in ('train', 'val', 'test')
    )

def all_processed_ready():
    return (
        any((PROCESSED_ALL / 'cloud').glob('*.npy')) or any((PROCESSED_ALL / 'clear').glob('*.npy'))
    ) and any((PROCESSED_ALL / 'masks').glob('*.npy'))

if not split_ready() or CFG['rebuild_processed_data']:
    if not all_processed_ready() or CFG['rebuild_processed_data']:
        preprocess_args = [
            sys.executable, PROJECT / 'src/data/preprocess_95cloud.py',
            '--data_dir', DATA_ROOT,
            '--out_dir', PROCESSED_ALL,
            '--keep-native-size',
            '--channels', '3',
            '--cloud_ratio_threshold', str(CFG['cloud_ratio_threshold']),
        ]
        invalid_values = [str(int(value)) for value in raw_audit['invalid_ground_truth_values']]
        if invalid_values:
            preprocess_args.extend(['--invalid-ground-truth-values', *invalid_values])
        if CFG['rebuild_processed_data']:
            preprocess_args.append('--force')
        run_command('Preprocess native RGB segmentation pairs', preprocess_args, cwd=RUN)

    split_args = [
        sys.executable, PROJECT / 'src/data/split_dataset.py',
        '--src_dir', PROCESSED_ALL,
        '--out_dir', PROCESSED,
        '--val_ratio', str(CFG['val_ratio']),
        '--test_ratio', str(CFG['test_ratio']),
        '--seed', str(CFG['seed']),
    ]
    if CFG['move_split']:
        split_args.append('--move')
    if CFG['rebuild_processed_data']:
        split_args.append('--force')
    run_command('Create scene-level split', split_args, cwd=RUN)
else:
    print('Using existing processed scene split:', PROCESSED)

split_manifest_path = PROCESSED / 'scene_split_manifest.json'
split_lineage_path = PROCESSED / 'scene_split_lineage.json'
if not split_ready() or not split_lineage_path.is_file():
    raise RuntimeError('Processed scene split or lineage manifest is incomplete')
split_manifest = json.loads(split_manifest_path.read_text(encoding='utf-8'))
split_lineage = json.loads(split_lineage_path.read_text(encoding='utf-8'))
print('Processed split lineage:', split_lineage['lineage_id'])
disk_report('After preprocessing and split')


## 9. Validate the segmentation dataset and derive train-only statistics

This checks split leakage, target values, validity semantics, tensor finiteness, and native spatial shapes. The train-only RGB statistics are recorded for the data audit. The released MVP `InputSpec` is still pinned to its dtype-range normalization, so these values are not silently applied to this run.


In [ ]:
from src.data.segmentation_dataset import SegmentationDataset

split_names = ('train', 'val', 'test')
scene_sets = {split: set(split_manifest[split]['scenes']) for split in split_names}
for index, left in enumerate(split_names):
    for right in split_names[index + 1:]:
        overlap = sorted(scene_sets[left] & scene_sets[right])
        if overlap:
            raise ValueError(f'Scene leakage between {left} and {right}: {overlap[:5]}')

dataset_summary = {}
channel_sum = np.zeros(3, dtype=np.float64)
channel_sumsq = np.zeros(3, dtype=np.float64)
train_valid_pixels = 0
for split in split_names:
    dataset = SegmentationDataset(PROCESSED / split, is_train=False, preserve_native_size=True)
    if len(dataset) != int(split_manifest[split]['image_count']):
        raise ValueError(f'{split} sample count differs from its split manifest')
    valid_pixels = 0
    cloud_pixels = 0
    source_pixels = 0
    spatial_shapes = set()
    source_scenes = set()
    for sample in dataset:
        image = sample['image']
        target = sample['mask']
        validity = sample['validity_mask']
        if image.dtype != torch.float32 or image.ndim != 3 or image.shape[0] != 3:
            raise TypeError(f'Invalid image tensor in {split}: {tuple(image.shape)} {image.dtype}')
        if not torch.isfinite(image).all() or target.shape != validity.shape or target.shape != image.shape[1:]:
            raise ValueError(f'Invalid image/mask/validity alignment in {split}')
        valid_values = target[validity]
        if valid_values.numel() and not torch.all((valid_values == 0) | (valid_values == 1)):
            raise ValueError(f'Valid target contains values outside 0/1 in {split}')
        if torch.any(target[~validity] != 255):
            raise ValueError(f'Invalid pixels are not encoded as ignore_index in {split}')
        source_pixels += int(target.numel())
        valid_pixels += int(validity.sum())
        cloud_pixels += int((target[validity] == 1).sum())
        spatial_shapes.add(tuple(int(value) for value in target.shape))
        source_scenes.add(sample['scene_id'])
        if split == 'train' and validity.any():
            values = image[:, validity].double().cpu().numpy()
            channel_sum += values.sum(axis=1)
            channel_sumsq += np.square(values).sum(axis=1)
            train_valid_pixels += values.shape[1]
    dataset_summary[split] = {
        'samples': len(dataset),
        'scene_count': len(source_scenes),
        'source_pixels': source_pixels,
        'valid_pixels': valid_pixels,
        'valid_pixel_ratio': valid_pixels / source_pixels if source_pixels else 0.0,
        'cloud_pixel_ratio_on_valid': cloud_pixels / valid_pixels if valid_pixels else None,
        'native_spatial_shapes': [list(shape) for shape in sorted(spatial_shapes)],
    }

if train_valid_pixels <= 0:
    raise RuntimeError('Train split contains no valid pixels')
train_mean = channel_sum / train_valid_pixels
train_variance = np.maximum(channel_sumsq / train_valid_pixels - np.square(train_mean), 0.0)
train_statistics = {
    'dataset_role': 'train',
    'valid_pixel_count': int(train_valid_pixels),
    'tensor_space': 'uint16 divided by 65535',
    'band_order': ['red', 'green', 'blue'],
    'mean': [float(value) for value in train_mean],
    'std': [float(value) for value in np.sqrt(train_variance)],
    'applied_to_this_run': False,
    'reason': 'The pinned MVP InputSpec permits dtype-range normalization only. A mean/std ablation requires a versioned InputSpec and runtime parity update.',
}
(RESULTS / 'dataset_validation.json').write_text(
    json.dumps(dataset_summary, indent=2, sort_keys=True), encoding='utf-8'
)
(RESULTS / 'train_rgb_statistics.json').write_text(
    json.dumps(train_statistics, indent=2, sort_keys=True), encoding='utf-8'
)
print(json.dumps(dataset_summary, indent=2, sort_keys=True))
print(json.dumps(train_statistics, indent=2, sort_keys=True))


## 10. Train SegFormer-B0

The repository entry point uses masked Cross-Entropy plus masked Soft Dice, native labels, bilinear logit upsampling, AMP on CUDA, and skips all-invalid batches. It currently selects its checkpoint by validation loss, which is recorded in the bundle; a release candidate still needs an owner-approved checkpoint-selection metric. The local model initializes its own weights, so this run is explicitly recorded as `not_loaded` for the external pretrained artifact.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before training SegFormer-B0 in Colab')

from src.train_segmentation import SegmentationTrainingConfig, train

checkpoint_path = CHECKPOINTS / f"segformer_b0_rgb_{CFG['run_mode']}.pth"
training_config = SegmentationTrainingConfig(
    epochs=CFG['epochs'],
    learning_rate=CFG['learning_rate'],
    weight_decay=CFG['weight_decay'],
    warmup_epochs=CFG['warmup_epochs'],
    early_stopping_patience=CFG['early_stopping_patience'],
    cross_entropy_weight=1.0,
    dice_weight=1.0,
    dice_epsilon=1e-6,
    ignore_index=255,
    seed=CFG['seed'],
    use_amp=CFG['use_amp'],
    batch_size=1,
    preserve_native_size=True,
)
checkpoint_metadata = {
    'run_mode': CFG['run_mode'],
    'model_task': segformer_manifest['model_task'],
    'model_release_id': segformer_manifest['model_release_id'],
    'source_revision': source_revision,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'input_spec_id': segformer_manifest['input_spec']['input_spec_id'],
    'checkpoint_selection_metric': 'validation_loss',
    'seed_count': 1,
    'seed_limitation': 'This notebook runs one seed. The evaluation bootstrap quantifies scene sampling uncertainty but does not replace a multi-seed final candidate run.',
    'class_mapping': {'clear': 0, 'cloud': 1},
    'pretrained_artifact': {
        'artifact_id': segformer_manifest['implementation']['pretrained_artifact_id'],
        'load_status': 'not_loaded',
        'reason': 'No verified pretrained-weight loader and checksum are present in the local implementation.',
    },
}
training_report = train(
    PROCESSED / 'train',
    PROCESSED / 'val',
    checkpoint_path,
    config=training_config,
    device='cuda',
    metadata=checkpoint_metadata,
)
if not checkpoint_path.is_file():
    raise RuntimeError('Training completed without writing a checkpoint')
checkpoint_sha256 = sha256_file(checkpoint_path)
drive_checkpoint_path = save_to_drive(
    checkpoint_path, DRIVE_CHECKPOINTS / checkpoint_path.name
)
drive_training_report_path = save_to_drive(
    checkpoint_path.with_suffix('.json'),
    DRIVE_RESULTS / checkpoint_path.with_suffix('.json').name,
)
(RESULTS / 'training_summary.json').write_text(
    json.dumps({
        **training_report,
        'checkpoint_sha256': checkpoint_sha256,
        'drive_checkpoint_path': str(drive_checkpoint_path),
        'drive_training_report_path': str(drive_training_report_path),
    }, indent=2, sort_keys=True),
    encoding='utf-8',
)
print('Checkpoint:', checkpoint_path)
print('Checkpoint SHA-256:', checkpoint_sha256)
print('Checkpoint copied to Drive:', drive_checkpoint_path)
print('Best validation loss:', training_report['best_validation_loss'])


## 11. Calibrate the pixel threshold on validation only

The evaluator sweeps candidate thresholds against validation predictions, maximizes Dice under the false-clear constraint, and writes a candidate `DecisionSpec`. The test split is not used by this cell.


In [ ]:
calibration_path = RESULTS / 'validation_calibration.json'
run_command(
    'Calibrate pixel threshold on validation',
    [
        sys.executable, PROJECT / 'src/eval_segmentation.py',
        '--test-dir', PROCESSED / 'val',
        '--model-path', checkpoint_path,
        '--dataset-role', 'validation',
        '--select-threshold',
        '--max-false-clear-rate', str(CFG['max_false_clear_rate']),
        '--threshold-start-bp', str(CFG['threshold_start_bp']),
        '--threshold-stop-bp', str(CFG['threshold_stop_bp']),
        '--threshold-step-bp', str(CFG['threshold_step_bp']),
        '--bootstrap-samples', str(CFG['bootstrap_samples']),
        '--bootstrap-seed', str(CFG['seed']),
        '--device', 'cuda',
        '--output', calibration_path,
    ],
    cwd=RUN,
)
calibration_report = json.loads(calibration_path.read_text(encoding='utf-8'))
locked_threshold_bp = int(calibration_report['threshold_bp'])
candidate_decision_spec = dict(segformer_manifest['decision_spec'])
candidate_decision_spec.update({
    'pixel_cloud_probability_threshold_bp': locked_threshold_bp,
    'false_clear_constraint_bp': int(round(CFG['max_false_clear_rate'] * 10000)),
    'calibration_id': f'95cloud-validation-{checkpoint_sha256[:12]}',
    'calibration_report_sha256': sha256_file(calibration_path),
    'dataset_role': 'validation',
    'split_lineage_id': split_lineage['lineage_id'],
})
locked_decision_path = CONTRACTS / 'candidate_decision_spec.json'
locked_decision_path.write_text(
    json.dumps(candidate_decision_spec, indent=2, sort_keys=True), encoding='utf-8'
)
print('Locked validation threshold (bp):', locked_threshold_bp)
print('Candidate decision spec:', locked_decision_path)


## 12. Run the frozen test evaluation once

This uses the locked validation threshold without a test sweep. It reports micro/macro scene metrics, boundary F1, coverage errors, and deterministic scene bootstrap intervals.


In [ ]:
test_report_path = RESULTS / 'test_evaluation.json'
run_command(
    'Evaluate frozen test split',
    [
        sys.executable, PROJECT / 'src/eval_segmentation.py',
        '--test-dir', PROCESSED / 'test',
        '--model-path', checkpoint_path,
        '--dataset-role', 'test',
        '--threshold-bp', str(locked_threshold_bp),
        '--bootstrap-samples', str(CFG['bootstrap_samples']),
        '--bootstrap-seed', str(CFG['seed']),
        '--device', 'cuda',
        '--output', test_report_path,
    ],
    cwd=RUN,
)
test_report = json.loads(test_report_path.read_text(encoding='utf-8'))
metrics = test_report['metrics']
coverage = test_report['coverage_metrics']
quality = acceptance_profile['quality']
def check_at_least(name, actual, expected):
    return {'actual': actual, 'expected': expected, 'passed': actual is not None and actual >= expected}

def check_at_most(name, actual, expected):
    return {'actual': actual, 'expected': expected, 'passed': actual is not None and actual <= expected}

quality_gates = {
    'cloud_iou': check_at_least('cloud_iou', metrics['cloud_iou'], quality['min_cloud_iou']),
    'cloud_dice': check_at_least('cloud_dice', metrics['cloud_dice'], quality['min_cloud_dice']),
    'cloud_recall': check_at_least('cloud_recall', metrics['cloud_recall'], quality['min_cloud_recall']),
    'false_clear_rate': check_at_most('false_clear_rate', metrics['false_clear_rate'], quality['max_false_clear_rate']),
    'boundary_f1': check_at_least('boundary_f1', metrics['boundary_f1'], quality['min_boundary_f1']),
    'coverage_mae_bp': check_at_most('coverage_mae_bp', coverage['coverage_mae_bp'], quality['max_coverage_mae_bp']),
    'coverage_p95_abs_error_bp': check_at_most(
        'coverage_p95_abs_error_bp', coverage['coverage_p95_abs_error_bp'], quality['max_coverage_p95_abs_error_bp']
    ),
    'valid_pixel_ratio': check_at_least(
        'valid_pixel_ratio', test_report['valid_pixel_ratio'], acceptance_profile['runtime']['min_valid_pixel_ratio']
    ),
}
quality_gate_report = {
    'run_mode': CFG['run_mode'],
    'checkpoint_sha256': checkpoint_sha256,
    'split_lineage_id': split_lineage['lineage_id'],
    'threshold_bp': locked_threshold_bp,
    'all_quality_gates_passed': all(item['passed'] for item in quality_gates.values()),
    'gates': quality_gates,
    'release_status': 'not_a_release',
    'release_blockers': [
        'This notebook does not create a pinned pretrained artifact.',
        'This notebook runs one training seed; a final candidate requires three seeds or a documented resource exception.',
        'The model manifest, calibration binding, target benchmark, and TensorRT parity are not mutated by this run.',
        'Release promotion remains subject to the integration-plan gates.',
    ],
}
quality_gate_path = RESULTS / 'quality_gate_report.json'
quality_gate_path.write_text(
    json.dumps(quality_gate_report, indent=2, sort_keys=True), encoding='utf-8'
)
print(json.dumps(quality_gate_report, indent=2, sort_keys=True))


## 13. Export the fixed runtime graph and check PyTorch/ONNX parity

The runtime graph stays fixed at batch 1 and `256 x 256`. A padded/cropped normalized validation tile is stored as a golden input together with raw PyTorch and ONNX logits. TensorRT parity must still run on the pinned target outside Colab.


In [ ]:
onnx_path = ARTIFACTS / f"segformer_b0_rgb_{CFG['run_mode']}.onnx"
run_command(
    'Export fixed-shape ONNX',
    [
        sys.executable, PROJECT / 'src/export_segformer_onnx.py',
        '--checkpoint', checkpoint_path,
        '--output', onnx_path,
    ],
    cwd=RUN,
)

import onnxruntime as ort
from src.models.segformer_b0 import get_segformer_b0

golden_dataset = SegmentationDataset(PROCESSED / 'val', is_train=False, preserve_native_size=True)
golden_sample = golden_dataset[0]['image'].cpu().numpy()
golden_input = np.zeros((1, 3, 256, 256), dtype=np.float32)
copy_height = min(256, golden_sample.shape[1])
copy_width = min(256, golden_sample.shape[2])
golden_input[0, :, :copy_height, :copy_width] = golden_sample[:, :copy_height, :copy_width]

model = get_segformer_b0().eval()
checkpoint = torch.load(checkpoint_path, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
with torch.inference_mode():
    pytorch_logits = model(torch.from_numpy(golden_input)).cpu().numpy()
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
onnx_logits = session.run(['logits'], {'input': golden_input})[0]
if pytorch_logits.shape != (1, 2, 64, 64) or onnx_logits.shape != (1, 2, 64, 64):
    raise ValueError(f'Unexpected fixed-shape logits: {pytorch_logits.shape}, {onnx_logits.shape}')
difference = np.abs(pytorch_logits.astype(np.float32) - onnx_logits.astype(np.float32))
onnx_tolerance = float(acceptance_profile['parity']['pytorch_onnx'])
parity_report = {
    'input_shape': list(golden_input.shape),
    'output_shape': list(pytorch_logits.shape),
    'max_abs_difference': float(difference.max()),
    'mean_abs_difference': float(difference.mean()),
    'tolerance': onnx_tolerance,
    'passed': bool(np.allclose(pytorch_logits, onnx_logits, rtol=0.0, atol=onnx_tolerance)),
    'tensorrt': {
        'status': 'not_run',
        'reason': 'TensorRT parity must run on the pinned deployment target, not on a generic Colab GPU.',
    },
}
np.save(ARTIFACTS / 'golden_input.npy', golden_input)
np.save(ARTIFACTS / 'golden_pytorch_logits.npy', pytorch_logits)
np.save(ARTIFACTS / 'golden_onnx_logits.npy', onnx_logits)
parity_path = RESULTS / 'pytorch_onnx_parity.json'
parity_path.write_text(json.dumps(parity_report, indent=2, sort_keys=True), encoding='utf-8')
if not parity_report['passed']:
    raise AssertionError(f'PyTorch/ONNX parity failed: {parity_report}')
print(json.dumps(parity_report, indent=2, sort_keys=True))


## 14. Package the evidence bundle

Only model/reports/contracts/golden vectors are zipped. The raw and processed datasets stay out of the download archive.


In [ ]:
environment_report = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'packages': {
        name: importlib.metadata.version(name)
        for name in ('numpy', 'torch', 'tifffile', 'onnx', 'onnxruntime', 'PyYAML', 'kaggle')
    },
}
if torch.cuda.is_available():
    environment_report['gpu_name'] = torch.cuda.get_device_name(0)
    try:
        environment_report['nvidia_smi'] = subprocess.check_output(['nvidia-smi', '-q'], text=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        environment_report['nvidia_smi'] = 'unavailable'
environment_path = RESULTS / 'environment.json'
environment_path.write_text(json.dumps(environment_report, indent=2, sort_keys=True), encoding='utf-8')

bundle_files = [
    checkpoint_path,
    checkpoint_path.with_suffix('.json'),
    RESULTS / 'source_provenance.json',
    RESULTS / 'raw_dataset_audit.json',
    RESULTS / 'dataset_validation.json',
    RESULTS / 'train_rgb_statistics.json',
    RESULTS / 'training_summary.json',
    calibration_path,
    test_report_path,
    quality_gate_path,
    parity_path,
    environment_path,
    locked_decision_path,
    split_manifest_path,
    split_lineage_path,
    onnx_path,
    onnx_path.with_suffix('.onnx.json'),
    ARTIFACTS / 'golden_input.npy',
    ARTIFACTS / 'golden_pytorch_logits.npy',
    ARTIFACTS / 'golden_onnx_logits.npy',
]
for source in bundle_files:
    source = Path(source)
    if not source.is_file():
        raise FileNotFoundError(f'Expected artifact is missing: {source}')
    shutil.copy2(source, DELIVERABLES / source.name)

archive_base = CONTENT / 'segformer_95cloud_evidence_bundle'
archive_path = shutil.make_archive(str(archive_base), 'zip', RUN, 'deliverables')
for source in DELIVERABLES.iterdir():
    if source.is_file():
        save_to_drive(source, DRIVE_DELIVERABLES / source.name)
drive_archive_path = save_to_drive(archive_path, DRIVE_ROOT / Path(archive_path).name)
print('Evidence bundle:', archive_path)
print('Evidence bundle copied to Drive:', drive_archive_path)

from google.colab import files
files.download(archive_path)

if CFG['cleanup_local_after_bundle']:
    remove_content_path(PROCESSED)
    disk_report('After local processed-data cleanup')
